# 🧠 SENN — Self-Expanding Neural Network
### *The AI Child — Live Training Monitor*

Run each cell **top to bottom**. The brain visualizer appears inline in Cell 3.

| Cell | What it does |
|------|--------------|
| 1 | Clone repo & install deps |
| 2 | Patch brain.html for Colab (one-time) |
| 3 | Display live brain visualizer |
| 4 | Start training in background |
| 5 | Stream logs + wait for completion |

In [ ]:
# ── Cell 1: Clone repo and install dependencies ─────
!git clone https://github.com/Bindkushal/The-Child.git 2>/dev/null || \
    (cd The-Child && git pull)

%cd The-Child

!pip install torch gitpython numpy --quiet

print('✓ Ready')

In [ ]:
# ── Cell 2: Patch brain.html for Colab (one-time) ────
# brain.html normally fetches /api/state from a local server.
# In Colab we push state via Javascript instead — no server needed.

OLD = (
    "    const r = await fetch('/api/state',{cache:'no-store'});\n"
    "    if (!r.ok) throw 0;\n"
    "    const d = await r.json();\n"
    "    if (!d.current || d.current.step===0) throw 0;"
)

NEW = (
    "    // Colab mode: state pushed via window.__sennState\n"
    "    const d = window.__sennState;\n"
    "    if (!d || !d.current || d.current.step === 0) throw 0;"
)

with open('brain.html', 'r') as f:
    html = f.read()

if OLD in html:
    html = html.replace(OLD, NEW)
    with open('brain_colab.html', 'w') as f:
        f.write(html)
    print('✓ brain_colab.html written')
else:
    print('⚠ Pattern not found — copying original as fallback')
    import shutil
    shutil.copy('brain.html', 'brain_colab.html')

In [ ]:
# ── Cell 3: Display live brain visualizer ──────────
import threading, time, json
from IPython.display import display, HTML, Javascript

# Render brain.html inline (starts in DEMO mode — goes LIVE when training starts)
with open('brain_colab.html') as f:
    display(HTML(f.read()))

# ─ State pusher: reads live_state.json every 500ms → injects into page ─
_pusher_running = True

def _push_state_loop():
    while _pusher_running:
        try:
            with open('live_state.json') as f:
                raw = f.read()
            # Push into the page — brain.html poll() reads window.__sennState
            display(Javascript(f'window.__sennState = {raw};'))
        except FileNotFoundError:
            pass  # file not written yet, keep waiting
        except Exception:
            pass
        time.sleep(0.5)

_pusher = threading.Thread(target=_push_state_loop, daemon=True)
_pusher.start()
print('✓ Visualizer live — run Cell 4 to start training')

In [ ]:
# ── Cell 4: Start training in background thread ──────
import threading, sys

# Reload modules cleanly (safe to re-run this cell)
for mod in ['dynamic_net', 'growth_controller', 'memory_manager',
            'github_self_modifier', 'live_state_writer', 'train']:
    sys.modules.pop(mod, None)

import train as _train_module

_training_done = threading.Event()
_trained_net   = [None]

def _run_training():
    try:
        _trained_net[0] = _train_module.train()
    except Exception as e:
        print(f'\n[Training error] {e}')
        import traceback; traceback.print_exc()
    finally:
        _training_done.set()

_t = threading.Thread(target=_run_training, daemon=True)
_t.start()
print('✓ Training started in background')
print('  Watch the brain visualizer above ↑')
print('  Dot turns GREEN = live data flowing')

In [ ]:
# ── Cell 5: Stream logs + wait for completion (optional) ──
# This cell blocks until training finishes.
# The visualizer above keeps updating regardless.

try:
    _training_done.wait()
except NameError:
    print('Run Cell 4 first to start training.')
    raise

# Stop the state pusher
_pusher_running = False

net = _trained_net[0]
if net:
    print('\n' + '='*52)
    print('  TRAINING COMPLETE')
    print('='*52)
    arch = net.get_architecture()
    print(f"  Final net    : {net}")
    print(f"  Total params : {arch['total_params']:,}")
    print(f"  Growth events: {arch['growth_events']}")
    print()
    print('Files saved:')
    print('  senn_final.pt         ← trained weights')
    print('  training_log.json     ← full metrics history')
    print('  growth_journal.json   ← every growth event explained')